# NB-5 — W-1: second task under the same protocol (WinoGrande, DeBERTa-v3-large)

Implements `FMI_manifest_W1_second_task_WinoGrande.md`. Five seeds (13, 42, 71, 7595, 1676) × seven methods:
frozen head-only, token-wise MLP, LRAA, token mixer, LoRA r = 11; full fine-tuning with LRAA and without adapter.
Locked BoolQ hyperparameters; internal split by twin cluster; official dev (1,267) as evaluation set, scored once per
selected checkpoint; three-state intervention; D4; twin-cluster bootstrap; prespecified rules.

**First run with `SMOKE_TEST = True`** (5-minute pipeline check on a subsample, any GPU), then set it to `False` and run
the full study on an A100 (about 4 hours; restart-safe per seed × method, progress on Drive). The runtime is not deleted.

In [ ]:
import time
from datetime import datetime, timezone
RUN_STARTED_PERF = time.perf_counter(); RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
from google.colab import drive
drive.mount('/content/drive'); print('Run started (UTC):', RUN_STARTED_UTC)

In [ ]:
import subprocess, sys
packages = ['pandas==2.2.3', 'transformers==4.57.6', 'huggingface-hub==0.36.2', 'accelerate>=1.10,<2', 'sentencepiece>=0.2,<0.3', 'protobuf>=5.29,<6', 'scikit-learn==1.6.1', 'requests']
proc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *packages], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
if proc.returncode: print(proc.stdout); raise RuntimeError('Dependency installation failed.')
print('Dependencies installed.')

In [ ]:
import gc, hashlib, io, json, math, os, random, shutil, warnings, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
warnings.filterwarnings('ignore', category=FutureWarning); os.environ['TOKENIZERS_PARALLELISM'] = 'false'
try:
    from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not torch.cuda.is_available(): raise RuntimeError('GPU required.')
DEVICE = torch.device('cuda'); AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16; GPU_NAME = torch.cuda.get_device_name(0)
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True; torch.set_float32_matmul_precision('high')

SMOKE_TEST = True            # True: tiny subsample, 1 epoch, seeds 13 only — pipeline check. False: full study.
MANIFEST_ID = 'FMI_manifest_W1_second_task_WinoGrande.md'
DATA_SEED = 2026; SEEDS = [13, 42, 71, 7595, 1676]; EPOCHS = 3; MAX_LENGTH = 512
LATENT_RANK, LATENT_HEADS, MLP_BOTTLENECK = 256, 8, 512; MIXER_RANK, MIXER_KERNEL, MIXER_DILATION = 496, 32, 16; LORA_R, LORA_ALPHA, LORA_DROPOUT = 11, 22, 0.10
BATCH_SIZE, EVAL_BATCH_SIZE, GRAD_ACCUM_STEPS = 2, 8, 8; LR_FROZEN = 2e-4; ENCODER_LR = 8e-6; ADAPTER_SCORER_LR = 2e-4; WEIGHT_DECAY = 0.01; WARMUP_RATIO = 0.06
BOOTSTRAP_SEED, BOOTSTRAP_REPLICATES = 20260930, 10000
FROZEN_METHODS = ['head', 'mlp', 'lcca', 'token_mixer']; ALL_METHODS = FROZEN_METHODS + ['lora', 'fullft_lraa', 'fullft_none']
if SMOKE_TEST: SEEDS = [13]; EPOCHS = 1

DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/Colab Notebooks/NLP Modeller/deberta-v3-large'); LOCAL_MODEL_DIR = Path('/content/models/deberta-v3-large')
DRIVE_ROOT = Path('/content/drive/MyDrive/FMI') / ('NB5_winogrande_smoke' if SMOKE_TEST else 'NB5_winogrande'); PROGRESS = DRIVE_ROOT / 'progress'; STATES = DRIVE_ROOT / 'states'
WORK = Path('/content/fmi_nb5'); RESULT_DIR = WORK / 'results'; DATA_DIR = WORK / 'data'
for p in (WORK, RESULT_DIR, DATA_DIR, PROGRESS, STATES): p.mkdir(parents=True, exist_ok=True)
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(64 * 1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()
required = ['config.json', 'spm.model', 'tokenizer_config.json']
if any(not (DRIVE_MODEL_DIR / n).is_file() for n in required) or not any((DRIVE_MODEL_DIR / n).is_file() for n in ('model.safetensors', 'pytorch_model.bin')):
    raise FileNotFoundError(f'Incomplete base model folder: {DRIVE_MODEL_DIR}')
shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR, dirs_exist_ok=True); MODEL_NAME = str(LOCAL_MODEL_DIR)
print('GPU:', GPU_NAME, '| AMP', AMP_DTYPE, '| torch', torch.__version__, '| SMOKE_TEST =', SMOKE_TEST)

In [ ]:
# Data: WinoGrande 1.1 (size l) from the official archive; twin clusters; grouped internal split; official dev as evaluation set.
WG_URL = 'https://storage.googleapis.com/ai2-mosaic/public/winogrande/winogrande_1.1.zip'
zip_local = DATA_DIR / 'winogrande_1.1.zip'
if not zip_local.is_file():
    r = requests.get(WG_URL, timeout=120); r.raise_for_status(); zip_local.write_bytes(r.content)
WG_SHA = sha256_file(zip_local)
def read_split(zf, name):
    rows = [json.loads(l) for l in zf.read(f'winogrande_1.1/{name}.jsonl').decode('utf-8').splitlines() if l.strip()]
    labels = zf.read(f'winogrande_1.1/{name}-labels.lst').decode('utf-8').split()
    assert len(labels) == len(rows)
    out = []
    for r_, lab in zip(rows, labels):
        assert r_['sentence'].count('_') == 1, r_['qID']
        out.append({'qID': r_['qID'], 'cluster': r_['qID'].rsplit('-', 1)[0], 'sentence': r_['sentence'], 'option1': r_['option1'], 'option2': r_['option2'], 'gold': int(lab) - 1})
    return out
with zipfile.ZipFile(zip_local) as zf:
    train_all = read_split(zf, 'train_l'); dev = read_split(zf, 'dev')
assert len(train_all) == 10234 and len(dev) == 1267, (len(train_all), len(dev))
seed_everything(DATA_SEED)
groups = np.array([r_['cluster'] for r_ in train_all]); labels = np.array([r_['gold'] for r_ in train_all])
splitter = GroupShuffleSplit(n_splits=80, test_size=0.10, random_state=DATA_SEED); best = None
for tr, va in splitter.split(np.zeros(len(labels)), labels, groups):
    score = abs(len(va) / len(labels) - 0.10) + abs(labels[va].mean() - labels.mean())
    if best is None or score < best[0]: best = (score, tr, va)
_, train_idx, val_idx = best
train_rows = [train_all[i] for i in train_idx]; val_rows = [train_all[i] for i in val_idx]
assert set(r_['cluster'] for r_ in train_rows).isdisjoint(set(r_['cluster'] for r_ in val_rows))
if SMOKE_TEST: train_rows, val_rows, dev = train_rows[:160], val_rows[:64], dev[:200]
DEV_GOLD = np.array([r_['gold'] for r_ in dev]); _cl = {c: i for i, c in enumerate(sorted(set(r_['cluster'] for r_ in dev)))}
DEV_CLUSTER = np.array([_cl[r_['cluster']] for r_ in dev]); N_CLUSTERS = len(_cl)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, use_fast=True)
def fill(row, opt): return row['sentence'].replace('_', opt)
def encode_rows(rows):
    c1 = tokenizer([fill(r_, r_['option1']) for r_ in rows], truncation=True, max_length=MAX_LENGTH)
    c2 = tokenizer([fill(r_, r_['option2']) for r_ in rows], truncation=True, max_length=MAX_LENGTH)
    return [{'c': [{'input_ids': c1['input_ids'][i], 'attention_mask': c1['attention_mask'][i]}, {'input_ids': c2['input_ids'][i], 'attention_mask': c2['attention_mask'][i]}], 'gold': rows[i]['gold'], 'index': i} for i in range(len(rows))]
ENC = {'train': encode_rows(train_rows), 'val': encode_rows(val_rows), 'dev': encode_rows(dev)}
class PairDataset(Dataset):
    def __init__(self, items, training=False, seed=0, reverse=False): self.items, self.training, self.seed, self.epoch, self.reverse = items, training, int(seed), 0, reverse
    def set_epoch(self, e): self.epoch = int(e)
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        it = self.items[i]; swap = self.reverse
        if self.training: swap = random.Random(self.seed * 1000003 + self.epoch * 10007 + i).random() < 0.5
        order = [1, 0] if swap else [0, 1]
        return {'candidates': [it['c'][k] for k in order], 'semantics': order, 'label': order.index(it['gold']), 'gold': it['gold'], 'index': it['index']}
def collate(batch):
    flat = [c for b in batch for c in b['candidates']]; padded = tokenizer.pad(flat, padding=True, return_tensors='pt'); bsz = len(batch)
    return {'input_ids': padded['input_ids'].view(bsz, 2, -1), 'attention_mask': padded['attention_mask'].view(bsz, 2, -1), 'semantics': torch.tensor([b['semantics'] for b in batch]),
            'labels': torch.tensor([b['label'] for b in batch]), 'gold': torch.tensor([b['gold'] for b in batch]), 'index': torch.tensor([b['index'] for b in batch])}
def loader(split, seed, training=False, reverse=False):
    ds = PairDataset(ENC[split], training=training, seed=seed, reverse=reverse); g = torch.Generator().manual_seed(seed)
    return ds, DataLoader(ds, batch_size=BATCH_SIZE if training else EVAL_BATCH_SIZE, shuffle=training, collate_fn=collate, num_workers=2, pin_memory=True, generator=g)
n1 = int(sum(r_['gold'] for r_ in train_rows)); n0 = len(train_rows) - n1
CLASS_WEIGHT = torch.tensor([len(train_rows) / (2 * max(n0, 1)), len(train_rows) / (2 * max(n1, 1))], dtype=torch.float32, device=DEVICE)
audit = {'dataset': 'WinoGrande 1.1 size l', 'archive_sha256': WG_SHA, 'train_internal': len(train_rows), 'val_internal': len(val_rows), 'dev': len(dev), 'dev_twin_clusters': int(N_CLUSTERS), 'dev_gold_rate_option2': float(DEV_GOLD.mean()), 'class_weight': CLASS_WEIGHT.tolist(), 'smoke_test': SMOKE_TEST}
(RESULT_DIR / 'data_audit.json').write_text(json.dumps(audit, indent=2)); print(audit)

In [ ]:
# Modules (exact BoolQ-study code; three intervention states), LoRA wrapper, scorers.
class GatedMLPAdapter(nn.Module):
    def __init__(self, h, bottleneck, dropout=0.1):
        super().__init__(); self.down = nn.Linear(h, bottleneck, bias=False); self.up = nn.Linear(bottleneck, h, bias=False); self.norm = nn.LayerNorm(h); self.dropout = nn.Dropout(dropout)
        self.alpha_logit = nn.Parameter(torch.tensor(math.log(0.1 / 0.9), dtype=torch.float32))
    @property
    def gate(self): return torch.sigmoid(self.alpha_logit)
    def branch(self, hidden, mask): return self.up(F.gelu(self.down(hidden)))
    def forward(self, hidden, mask, state='on'):
        if state == 'off': return hidden
        if state == 'norm_only': return self.norm(hidden)
        return self.norm(hidden + self.gate * self.dropout(self.branch(hidden, mask)))
class LatentRelationAdapter(nn.Module):
    def __init__(self, h, rank, heads, dropout=0.1):
        super().__init__(); assert rank % heads == 0; self.rank, self.heads, self.head_dim = rank, heads, rank // heads
        self.q = nn.Linear(h, rank, bias=False); self.k = nn.Linear(h, rank, bias=False); self.v = nn.Linear(h, rank, bias=False); self.out = nn.Linear(rank, h, bias=False)
        self.norm = nn.LayerNorm(h); self.dropout = nn.Dropout(dropout); self.alpha_logit = nn.Parameter(torch.tensor(math.log(0.1 / 0.9), dtype=torch.float32))
    @property
    def gate(self): return torch.sigmoid(self.alpha_logit)
    def branch(self, hidden, mask):
        b, n, _ = hidden.shape; split = lambda x: x.view(b, n, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = split(self.q(hidden)), split(self.k(hidden)), split(self.v(hidden))
        sc = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim); sc = sc.masked_fill(mask[:, None, None, :] == 0, torch.finfo(sc.dtype).min)
        z = torch.matmul(torch.softmax(sc, dim=-1), v).transpose(1, 2).contiguous().view(b, n, self.rank); return self.out(z)
    def forward(self, hidden, mask, state='on'):
        if state == 'off': return hidden
        if state == 'norm_only': return self.norm(hidden)
        return self.norm(hidden + self.gate * self.dropout(self.branch(hidden, mask)))
class DilatedConvMixerAdapter(nn.Module):
    def __init__(self, h=1024, rank=MIXER_RANK, kernel=MIXER_KERNEL, dilation=MIXER_DILATION):
        super().__init__(); self.depthwise = nn.Conv1d(h, h, kernel_size=kernel, dilation=dilation, groups=h, bias=False); self.down = nn.Linear(h, rank, bias=False); self.up = nn.Linear(rank, h, bias=False)
        self.norm = nn.LayerNorm(h); self.dropout = nn.Dropout(0.10); self.gate_param = nn.Parameter(torch.zeros(())); self.total_pad = dilation * (kernel - 1)
    @property
    def gate(self): return torch.tanh(self.gate_param)
    def branch(self, hidden, mask):
        m = mask.unsqueeze(-1).to(hidden.dtype); x = hidden * m; left = self.total_pad // 2; right = self.total_pad - left
        mixed = self.depthwise(F.pad(x.transpose(1, 2), (left, right))).transpose(1, 2) * m; return self.up(F.gelu(self.down(mixed)))
    def forward(self, hidden, mask, state='on'):
        if state == 'off': return hidden
        if state == 'norm_only': return self.norm(hidden)
        return self.norm(hidden + self.gate * self.dropout(self.branch(hidden, mask)))
class FrozenFeatureScorer(nn.Module):
    def __init__(self, mode, h=1024):
        super().__init__(); self.mode = mode
        self.adapter = {'head': None, 'mlp': GatedMLPAdapter(h, MLP_BOTTLENECK), 'lcca': LatentRelationAdapter(h, LATENT_RANK, LATENT_HEADS), 'token_mixer': DilatedConvMixerAdapter(h)}[mode]
        self.dropout = nn.Dropout(0.15); self.scorer = nn.Linear(h, 1)
    def forward(self, hidden, mask, state='on'):
        if self.adapter is not None: hidden = self.adapter(hidden, mask, state=state)
        return self.scorer(self.dropout(hidden[:, 0]))
    @torch.no_grad()
    def branch_ratio_cls(self, hidden, mask):
        z = self.adapter.branch(hidden, mask)[:, 0].float() * self.adapter.gate.float(); return z.norm(dim=-1) / hidden[:, 0].float().norm(dim=-1).clamp_min(1e-6)
class LoRALinear(nn.Module):
    def __init__(self, base, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT):
        super().__init__(); self.base = base; self.r = r; self.scaling = alpha / r; self.dropout = nn.Dropout(dropout); self.enabled = True
        self.lora_A = nn.Parameter(torch.zeros(r, base.in_features)); self.lora_B = nn.Parameter(torch.zeros(base.out_features, r)); nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
    def forward(self, x):
        out = self.base(x)
        if self.enabled: out = out + F.linear(F.linear(self.dropout(x), self.lora_A), self.lora_B) * self.scaling
        return out
def inject_lora(encoder):
    mods = []
    for layer in encoder.encoder.layer:
        att = layer.attention.self; att.query_proj = LoRALinear(att.query_proj); att.value_proj = LoRALinear(att.value_proj); mods += [att.query_proj, att.value_proj]
    return mods
class LoRAScorer(nn.Module):
    def __init__(self, encoder):
        super().__init__(); self.encoder = encoder; self.lora_modules = inject_lora(encoder); self.dropout = nn.Dropout(0.15); self.scorer = nn.Linear(1024, 1)
        for n, p in self.encoder.named_parameters(): p.requires_grad = ('lora_A' in n or 'lora_B' in n)
    def set_lora(self, enabled):
        for m in self.lora_modules: m.enabled = enabled
    def forward(self, ids, mask, state='on'):
        self.set_lora(state == 'on'); h = self.encoder(input_ids=ids, attention_mask=mask).last_hidden_state; self.set_lora(True); return self.scorer(self.dropout(h[:, 0]))
class FullFineTuned(nn.Module):
    def __init__(self, arm):
        super().__init__(); self.arm = arm; self.encoder = AutoModel.from_pretrained(MODEL_NAME, token=HF_TOKEN)
        self.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        self.adapter = LatentRelationAdapter(1024, LATENT_RANK, LATENT_HEADS) if arm == 'lraa' else None; self.dropout = nn.Dropout(0.15); self.scorer = nn.Linear(1024, 1)
    def score_hidden(self, hidden, mask, state='on'):
        if self.adapter is not None: hidden = self.adapter(hidden, mask, state=state)
        return self.scorer(self.dropout(hidden[:, 0]))
    def forward(self, ids, mask, state='on'): return self.score_hidden(self.encoder(input_ids=ids, attention_mask=mask).last_hidden_state, mask, state)
    @torch.no_grad()
    def branch_ratio_cls(self, hidden, mask):
        z = self.adapter.branch(hidden, mask)[:, 0].float() * self.adapter.gate.float(); return z.norm(dim=-1) / hidden[:, 0].float().norm(dim=-1).clamp_min(1e-6)
_probe = FrozenFeatureScorer('lcca'); _m = FrozenFeatureScorer('mlp'); _x = FrozenFeatureScorer('token_mixer')
PARAM_COUNTS = {'head_scorer': sum(p.numel() for p in _probe.scorer.parameters()), 'lcca': sum(p.numel() for p in _probe.adapter.parameters()), 'mlp': sum(p.numel() for p in _m.adapter.parameters()), 'token_mixer': sum(p.numel() for p in _x.adapter.parameters()), 'lora': 24 * 2 * LORA_R * 2048}
assert PARAM_COUNTS['lcca'] == PARAM_COUNTS['mlp'] == PARAM_COUNTS['token_mixer'] == 1050625 and PARAM_COUNTS['lora'] == 1081344
del _probe, _m, _x; print(PARAM_COUNTS)

In [ ]:
# Evaluation and training routines (locked settings); restart-safe per (seed, method).
def metrics(gold, pred, prob, loss):
    return {'loss': float(loss), 'accuracy': accuracy_score(gold, pred), 'balanced_accuracy': balanced_accuracy_score(gold, pred), 'macro_f1': f1_score(gold, pred, average='macro', zero_division=0),
            'predicted_option2_rate': float(np.mean(pred)), 'log_loss': float(log_loss(gold, np.clip(prob, 1e-7, 1 - 1e-7), labels=[0, 1]))}
@torch.no_grad()
def evaluate(model_kind, model, encoder, loader_, states, collect_branch=False):
    model.eval(); tot = {st: {'loss': 0.0, 'gold': [], 'pred': [], 'prob2': [], 'index': []} for st in states}; ratios = []
    for batch in loader_:
        ids = batch['input_ids'].to(DEVICE); mask = batch['attention_mask'].to(DEVICE); labels = batch['labels'].to(DEVICE); sem = batch['semantics']; gold = batch['gold'].numpy(); bsz, nopt, L = ids.shape
        fi, fm = ids.view(bsz * nopt, L), mask.view(bsz * nopt, L)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            if model_kind == 'lora':
                for st in states:
                    logits = model(fi, fm, state=st).view(bsz, nopt).float(); _record(tot[st], logits, labels, sem, gold, batch['index'], bsz)
                continue
            hidden = encoder(input_ids=fi, attention_mask=fm).last_hidden_state if model_kind == 'frozen' else model.encoder(input_ids=fi, attention_mask=fm).last_hidden_state
            for st in states:
                logits = (model(hidden, fm, state=st) if model_kind == 'frozen' else model.score_hidden(hidden, fm, state=st)).view(bsz, nopt).float(); _record(tot[st], logits, labels, sem, gold, batch['index'], bsz)
            if collect_branch: ratios.extend(model.branch_ratio_cls(hidden, fm).cpu().numpy().tolist())
    out = {}
    for st, v in tot.items():
        g, p, pr = np.asarray(v['gold']), np.asarray(v['pred']), np.asarray(v['prob2']); order = np.argsort(np.asarray(v['index']))
        out[st] = {**metrics(g, p, pr, v['loss'] / len(g)), 'gold': g[order], 'pred': p[order], 'prob2': pr[order]}
    return out, (float(np.mean(ratios)) if ratios else None)
def _record(t, logits, labels, sem, gold, index, bsz):
    t['loss'] += float(F.cross_entropy(logits, labels, reduction='sum')); probs = torch.softmax(logits, dim=-1).cpu(); pi = logits.argmax(dim=-1).cpu()
    t['pred'].extend(sem.gather(1, pi[:, None]).squeeze(1).numpy().tolist()); t['prob2'].extend(probs.gather(1, (sem == 1).long().argmax(dim=1, keepdim=True)).squeeze(1).numpy().tolist())
    t['gold'].extend(gold.tolist()); t['index'].extend(index.numpy().tolist())
def cpu_state(m): return {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
def sched(optimizer, n_batches):
    upd = math.ceil(n_batches / GRAD_ACCUM_STEPS) * EPOCHS; return get_linear_schedule_with_warmup(optimizer, int(upd * WARMUP_RATIO), upd)
def sel_key(m): return (m['balanced_accuracy'], m['macro_f1'], -m['loss'])

def train_frozen_quartet(seed, encoder):
    seed_everything(seed); models = {m: FrozenFeatureScorer(m).to(DEVICE) for m in FROZEN_METHODS}
    train_ds, train_ld = loader('train', seed, training=True); _, val_ld = loader('val', seed)
    opts = {m: torch.optim.AdamW(models[m].parameters(), lr=LR_FROZEN, weight_decay=WEIGHT_DECAY) for m in FROZEN_METHODS}; scheds = {m: sched(opts[m], len(train_ld)) for m in FROZEN_METHODS}
    best = {m: {'key': None, 'state': None, 'epoch': None, 'val': None} for m in FROZEN_METHODS}; hist = []
    for epoch in range(1, EPOCHS + 1):
        train_ds.set_epoch(epoch); [models[m].train() for m in FROZEN_METHODS]
        for step, batch in enumerate(train_ld, 1):
            ids = batch['input_ids'].to(DEVICE); mask = batch['attention_mask'].to(DEVICE); labels = batch['labels'].to(DEVICE); gold = batch['gold'].to(DEVICE); bsz, nopt, L = ids.shape; fi, fm = ids.view(bsz * nopt, L), mask.view(bsz * nopt, L)
            with torch.no_grad(), torch.amp.autocast('cuda', dtype=AMP_DTYPE): hidden = encoder(input_ids=fi, attention_mask=fm).last_hidden_state
            for m in FROZEN_METHODS:
                with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                    logits = models[m](hidden, fm, state='on').view(bsz, nopt); loss = (F.cross_entropy(logits.float(), labels, reduction='none') * CLASS_WEIGHT[gold]).mean()
                (loss / GRAD_ACCUM_STEPS).backward()
                if step % GRAD_ACCUM_STEPS == 0 or step == len(train_ld):
                    torch.nn.utils.clip_grad_norm_(models[m].parameters(), 1.0); opts[m].step(); opts[m].zero_grad(set_to_none=True); scheds[m].step()
            if step % 1000 == 0: print(f'  frozen quartet seed {seed} epoch {epoch} step {step}/{len(train_ld)}')
        for m in FROZEN_METHODS:
            ev, _ = evaluate('frozen', models[m], encoder, val_ld, ['on']); v = ev['on']; row = {'method': m, 'seed': seed, 'epoch': epoch, **{f'val_{k}': x for k, x in v.items() if k not in ('gold', 'pred', 'prob2')}}; hist.append(row)
            if best[m]['key'] is None or sel_key(v) > best[m]['key']: best[m] = {'key': sel_key(v), 'state': cpu_state(models[m]), 'epoch': epoch, 'val': {k: x for k, x in v.items() if k not in ('gold', 'pred', 'prob2')}}
            print({k: (round(x, 4) if isinstance(x, float) else x) for k, x in row.items()})
    for m in FROZEN_METHODS: models[m].load_state_dict(best[m]['state']); models[m].eval()
    return models, best, hist
def train_lora(seed):
    seed_everything(seed); enc = AutoModel.from_pretrained(MODEL_NAME, token=HF_TOKEN); model = LoRAScorer(enc).to(DEVICE)
    train_ds, train_ld = loader('train', seed, training=True); _, val_ld = loader('val', seed)
    params = [p for p in model.parameters() if p.requires_grad]; opt = torch.optim.AdamW(params, lr=LR_FROZEN, weight_decay=WEIGHT_DECAY); sc = sched(opt, len(train_ld)); best = {'key': None}; hist = []
    for epoch in range(1, EPOCHS + 1):
        train_ds.set_epoch(epoch); model.train()
        for step, batch in enumerate(train_ld, 1):
            ids = batch['input_ids'].to(DEVICE); mask = batch['attention_mask'].to(DEVICE); labels = batch['labels'].to(DEVICE); gold = batch['gold'].to(DEVICE); bsz, nopt, L = ids.shape
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                logits = model(ids.view(bsz * nopt, L), mask.view(bsz * nopt, L), state='on').view(bsz, nopt); loss = (F.cross_entropy(logits.float(), labels, reduction='none') * CLASS_WEIGHT[gold]).mean()
            (loss / GRAD_ACCUM_STEPS).backward()
            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_ld): torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); opt.zero_grad(set_to_none=True); sc.step()
            if step % 1000 == 0: print(f'  lora seed {seed} epoch {epoch} step {step}/{len(train_ld)}')
        ev, _ = evaluate('lora', model, None, val_ld, ['on']); v = ev['on']; row = {'method': 'lora', 'seed': seed, 'epoch': epoch, **{f'val_{k}': x for k, x in v.items() if k not in ('gold', 'pred', 'prob2')}}; hist.append(row); print({k: (round(x, 4) if isinstance(x, float) else x) for k, x in row.items()})
        if best['key'] is None or sel_key(v) > best['key']: best = {'key': sel_key(v), 'state': {k: t.detach().cpu().clone() for k, t in model.state_dict().items() if ('lora_A' in k or 'lora_B' in k or k.startswith('scorer'))}, 'epoch': epoch, 'val': {k: x for k, x in v.items() if k not in ('gold', 'pred', 'prob2')}}
    model.load_state_dict(best['state'], strict=False); model.eval(); return model, best, hist
def train_fullft(arm, seed):
    seed_everything(seed); model = FullFineTuned(arm).to(DEVICE); train_ds, train_ld = loader('train', seed, training=True); _, val_ld = loader('val', seed)
    head = list(model.scorer.parameters()) + (list(model.adapter.parameters()) if model.adapter is not None else [])
    opt = torch.optim.AdamW([{'params': list(model.encoder.parameters()), 'lr': ENCODER_LR}, {'params': head, 'lr': ADAPTER_SCORER_LR}], weight_decay=WEIGHT_DECAY); sc = sched(opt, len(train_ld)); best = {'key': None}; hist = []
    for epoch in range(1, EPOCHS + 1):
        train_ds.set_epoch(epoch); model.train()
        for step, batch in enumerate(train_ld, 1):
            ids = batch['input_ids'].to(DEVICE); mask = batch['attention_mask'].to(DEVICE); labels = batch['labels'].to(DEVICE); gold = batch['gold'].to(DEVICE); bsz, nopt, L = ids.shape
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                logits = model(ids.view(bsz * nopt, L), mask.view(bsz * nopt, L), state='on').view(bsz, nopt); loss = (F.cross_entropy(logits.float(), labels, reduction='none') * CLASS_WEIGHT[gold]).mean()
            if not torch.isfinite(loss): raise FloatingPointError('non-finite loss')
            (loss / GRAD_ACCUM_STEPS).backward()
            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_ld): torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); opt.zero_grad(set_to_none=True); sc.step()
            if step % 1000 == 0: print(f'  fullft_{arm} seed {seed} epoch {epoch} step {step}/{len(train_ld)}')
        ev, _ = evaluate('fullft', model, None, val_ld, ['on']); v = ev['on']; row = {'method': f'fullft_{"lraa" if arm == "lraa" else "none"}', 'seed': seed, 'epoch': epoch, **{f'val_{k}': x for k, x in v.items() if k not in ('gold', 'pred', 'prob2')}}; hist.append(row); print({k: (round(x, 4) if isinstance(x, float) else x) for k, x in row.items()})
        if best['key'] is None or sel_key(v) > best['key']: best = {'key': sel_key(v), 'state': cpu_state(model), 'epoch': epoch, 'val': {k: x for k, x in v.items() if k not in ('gold', 'pred', 'prob2')}}
    model.load_state_dict(best['state']); model.eval(); return model, best, hist

In [ ]:
# Runs: for each seed — frozen quartet (shared frozen forward), LoRA, full FT with LRAA, full FT without adapter. Dev scored once per selected checkpoint; results withheld until the end.
def prog_path(seed, method): return PROGRESS / f'{method}_seed{seed}.json'
def done(seed, method): return prog_path(seed, method).is_file()
def commit(seed, method, payload, state=None):
    if state is not None:
        sp = STATES / f'{method}_seed{seed}.pt'; torch.save(state, sp); payload['state_file'] = str(sp); payload['state_sha256'] = sha256_file(sp); payload['state_bytes'] = sp.stat().st_size
    prog_path(seed, method).write_text(json.dumps(payload, indent=1, default=float))
def dev_records(seed, method, ev, states, branch_ratio=None, gate=None):
    rec = {'seed': seed, 'method': method, 'states': {}, 'branch_ratio_cls': branch_ratio, 'gate_value': gate}
    for st in states:
        v = ev[st]; rec['states'][st] = {'metrics': {k: x for k, x in v.items() if k not in ('gold', 'pred', 'prob2')}, 'pred': v['pred'].tolist(), 'prob2': v['prob2'].tolist()}
    return rec
_, DEV_LD = loader('dev', 0); _, DEV_LD_REV = loader('dev', 0, reverse=True)
ENCODER = None
for seed in SEEDS:
    t_seed = time.time()
    if not all(done(seed, m) for m in FROZEN_METHODS):
        if ENCODER is None:
            ENCODER = AutoModel.from_pretrained(MODEL_NAME, token=HF_TOKEN).to(DEVICE).eval()
            for p in ENCODER.parameters(): p.requires_grad = False
        print(f'\n===== FROZEN QUARTET, seed {seed} =====')
        models, best, hist = train_frozen_quartet(seed, ENCODER)
        for m in FROZEN_METHODS:
            states = ['on'] if m == 'head' else ['on', 'norm_only', 'off']
            ev, br = evaluate('frozen', models[m], ENCODER, DEV_LD, states, collect_branch=(m != 'head')); ev_rev, _ = evaluate('frozen', models[m], ENCODER, DEV_LD_REV, ['on'])
            consistency = float(np.mean(ev['on']['pred'] == ev_rev['on']['pred'])); gate = None if m == 'head' else float(models[m].adapter.gate.detach().cpu())
            rec = dev_records(seed, m, ev, states, br, gate); rec.update({'best_epoch': best[m]['epoch'], 'best_val': best[m]['val'], 'history': [h for h in hist if h['method'] == m], 'option_order_consistency': consistency, 'gpu': GPU_NAME})
            commit(seed, m, rec, best[m]['state'])
        del models; gc.collect(); torch.cuda.empty_cache()
    if not done(seed, 'lora'):
        print(f'\n===== LoRA r={LORA_R}, seed {seed} ====='); model, best, hist = train_lora(seed)
        ev, _ = evaluate('lora', model, None, DEV_LD, ['on', 'off']); ev_rev, _ = evaluate('lora', model, None, DEV_LD_REV, ['on'])
        rec = dev_records(seed, 'lora', ev, ['on', 'off']); rec.update({'best_epoch': best['epoch'], 'best_val': best['val'], 'history': hist, 'option_order_consistency': float(np.mean(ev['on']['pred'] == ev_rev['on']['pred'])), 'gpu': GPU_NAME})
        commit(seed, 'lora', rec, best['state']); del model; gc.collect(); torch.cuda.empty_cache()
    for arm, name in (('lraa', 'fullft_lraa'), ('none', 'fullft_none')):
        if done(seed, name): continue
        if ENCODER is not None: del ENCODER; ENCODER = None; gc.collect(); torch.cuda.empty_cache()
        print(f'\n===== FULL FINE-TUNING ({arm}), seed {seed} ====='); model, best, hist = train_fullft(arm, seed)
        states = ['on', 'norm_only', 'off'] if arm == 'lraa' else ['on']
        ev, br = evaluate('fullft', model, None, DEV_LD, states, collect_branch=(arm == 'lraa')); ev_rev, _ = evaluate('fullft', model, None, DEV_LD_REV, ['on'])
        rec = dev_records(seed, name, ev, states, br, (float(model.adapter.gate.detach().cpu()) if arm == 'lraa' else None)); rec.update({'best_epoch': best['epoch'], 'best_val': best['val'], 'history': hist, 'option_order_consistency': float(np.mean(ev['on']['pred'] == ev_rev['on']['pred'])), 'gpu': GPU_NAME})
        commit(seed, name, rec, best['state'] if not SMOKE_TEST else None); del model; gc.collect(); torch.cuda.empty_cache()
    print(f'seed {seed} complete ({(time.time() - t_seed) / 60:.1f} min)')
print('All runs complete.')

In [ ]:
# Analysis: tables, prespecified rules, twin-cluster bootstrap, McNemar, D4.
R = {(s, m): json.loads(prog_path(s, m).read_text()) for s in SEEDS for m in ALL_METHODS}
def BA(s, m, st='on'): return 100 * R[(s, m)]['states'][st]['metrics']['balanced_accuracy']
def PRED(s, m, st='on'): return np.array(R[(s, m)]['states'][st]['pred'])
T = {2: 4.302652729911275, 4: 2.7764451051977987}
def ci(v):
    v = np.asarray(v, float); m = float(v.mean()); sd = float(v.std(ddof=1)) if len(v) > 1 else float('nan'); h = T.get(len(v) - 1, 4.302652729911275) * sd / math.sqrt(len(v)) if len(v) > 1 else float('nan'); return {'mean': m, 'sd': sd, 'ci95_low': m - h, 'ci95_high': m + h, 'values': [float(x) for x in v], 'positive_seeds': int((v > 0).sum())}
def exact_mcnemar_p(x, y):
    n = x + y
    if n == 0: return 1.0
    k = min(x, y); return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)
rows = []
for m in ALL_METHODS:
    for s in SEEDS:
        for st, v in R[(s, m)]['states'].items(): rows.append({'method': m, 'seed': s, 'state': st, 'best_epoch': R[(s, m)]['best_epoch'], **v['metrics'], 'option_order_consistency': R[(s, m)]['option_order_consistency'], 'gate_value': R[(s, m)]['gate_value'], 'branch_ratio_cls': R[(s, m)]['branch_ratio_cls']})
RUN = pd.DataFrame(rows); RUN.to_csv(RESULT_DIR / 'dev_run_results.csv', index=False)
pd.DataFrame([h for m in ALL_METHODS for s in SEEDS for h in R[(s, m)]['history']]).to_csv(RESULT_DIR / 'training_history.csv', index=False)
pred_rows = [{'method': m, 'seed': s, 'state': st, 'example_index': i, 'twin_cluster': int(DEV_CLUSTER[i]), 'gold': int(DEV_GOLD[i]), 'prediction': int(p), 'prob_option2': float(q)} for m in ALL_METHODS for s in SEEDS for st, v in R[(s, m)]['states'].items() for i, (p, q) in enumerate(zip(v['pred'], v['prob2']))]
pd.DataFrame(pred_rows).to_csv(RESULT_DIR / 'dev_predictions.csv', index=False)
# seed-level quantities
Q = {}
Q['lcca_minus_mlp'] = ci([BA(s, 'lcca') - BA(s, 'mlp') for s in SEEDS]); Q['lcca_minus_mixer'] = ci([BA(s, 'lcca') - BA(s, 'token_mixer') for s in SEEDS]); Q['lcca_minus_lora'] = ci([BA(s, 'lcca') - BA(s, 'lora') for s in SEEDS])
for m in ('mlp', 'lcca', 'token_mixer', 'fullft_lraa'):
    Q[f'A_{m}'] = ci([BA(s, m, 'on') - BA(s, m, 'norm_only') for s in SEEDS]); Q[f'B_{m}'] = ci([BA(s, m, 'norm_only') - BA(s, m, 'off') for s in SEEDS]); Q[f'on_off_{m}'] = ci([BA(s, m, 'on') - BA(s, m, 'off') for s in SEEDS])
Q['on_off_lora'] = ci([BA(s, 'lora', 'on') - BA(s, 'lora', 'off') for s in SEEDS])
Q['delta_A'] = ci([(BA(s, 'lcca', 'on') - BA(s, 'lcca', 'norm_only')) - (BA(s, 'fullft_lraa', 'on') - BA(s, 'fullft_lraa', 'norm_only')) for s in SEEDS])
Q['delta_on_off'] = ci([(BA(s, 'lcca', 'on') - BA(s, 'lcca', 'off')) - (BA(s, 'fullft_lraa', 'on') - BA(s, 'fullft_lraa', 'off')) for s in SEEDS])
Q['D2'] = ci([BA(s, 'fullft_lraa') - BA(s, 'fullft_none') for s in SEEDS]); Q['G_lcca'] = ci([BA(s, 'lcca') - BA(s, 'head') for s in SEEDS])
means = {m: {st: float(np.mean([BA(s, m, st) for s in SEEDS])) for st in R[(SEEDS[0], m)]['states']} for m in ALL_METHODS}
# optimization gate for full FT
best_val = pd.DataFrame([h for s in SEEDS for h in R[(s, 'fullft_lraa')]['history']]).groupby('seed')['val_balanced_accuracy'].max()
gate = {'all_seed_best_internal_val_BA_ge_0.60': bool((best_val >= 0.60).all()), 'all_enabled_eval_BA_above_chance': bool(all(BA(s, 'fullft_lraa') > 50 for s in SEEDS)), 'enabled_mean_above_frozen_head_only': bool(means['fullft_lraa']['on'] > means['head']['on'])}
gate['status'] = 'VALID' if all(gate.values()) else 'INVALID_OPTIMIZATION'
# rules
def rule3(q, min_gain=0.5): return {'mean_ge_threshold': q['mean'] >= min_gain, 'positive_all_seeds': q['positive_seeds'] == len(SEEDS), 'ci_low_gt_0': q['ci95_low'] > 0, 'met': (q['mean'] >= min_gain) and (q['positive_seeds'] == len(SEEDS)) and (q['ci95_low'] > 0)}
DEC = {'confirmation_lcca_vs_mlp': rule3(Q['lcca_minus_mlp']), 'attention_topology_lcca_vs_mixer': rule3(Q['lcca_minus_mixer']), 'training_regime': {**gate, 'delta_A_positive_all_seeds': Q['delta_A']['positive_seeds'] == len(SEEDS), 'delta_A_ci_low_gt_0': Q['delta_A']['ci95_low'] > 0, 'met': gate['status'] == 'VALID' and Q['delta_A']['positive_seeds'] == len(SEEDS) and Q['delta_A']['ci95_low'] > 0},
       'D2_wording': ('no measurable change' if Q['D2']['ci95_low'] <= 0 <= Q['D2']['ci95_high'] else ('LRAA adds to full fine-tuning' if Q['D2']['ci95_low'] > 0 else 'LRAA reduces full fine-tuning performance'))}
# twin-cluster bootstrap (shared resample across seeds/methods/states)
pos = DEV_GOLD == 1; cols, cidx = [], {}
def add(key, vec): cidx[key] = len(cols); cols.append(np.bincount(DEV_CLUSTER, weights=vec.astype(float), minlength=N_CLUSTERS))
add('npos', pos.astype(float)); add('nneg', (~pos).astype(float))
for m in ALL_METHODS:
    for s in SEEDS:
        for st in R[(s, m)]['states']:
            c = PRED(s, m, st) == DEV_GOLD; add((m, s, st, 'cp'), (c & pos).astype(float)); add((m, s, st, 'cn'), (c & ~pos).astype(float))
M = np.stack(cols, axis=1); rng = np.random.default_rng(BOOTSTRAP_SEED); acc = {k: [] for k in ('lcca_minus_mlp', 'lcca_minus_mixer', 'lcca_minus_lora', 'A_lcca', 'B_lcca', 'A_fullft_lraa', 'B_fullft_lraa', 'delta_A', 'D2')}
for _ in range(BOOTSTRAP_REPLICATES if not SMOKE_TEST else 200):
    idx = rng.integers(0, N_CLUSTERS, N_CLUSTERS); S = M[idx].sum(axis=0); ba = lambda m, s, st: 50 * (S[cidx[(m, s, st, 'cp')]] / S[cidx['npos']] + S[cidx[(m, s, st, 'cn')]] / S[cidx['nneg']])
    acc['lcca_minus_mlp'].append(np.mean([ba('lcca', s, 'on') - ba('mlp', s, 'on') for s in SEEDS])); acc['lcca_minus_mixer'].append(np.mean([ba('lcca', s, 'on') - ba('token_mixer', s, 'on') for s in SEEDS])); acc['lcca_minus_lora'].append(np.mean([ba('lcca', s, 'on') - ba('lora', s, 'on') for s in SEEDS]))
    Af = [ba('lcca', s, 'on') - ba('lcca', s, 'norm_only') for s in SEEDS]; At = [ba('fullft_lraa', s, 'on') - ba('fullft_lraa', s, 'norm_only') for s in SEEDS]
    acc['A_lcca'].append(np.mean(Af)); acc['B_lcca'].append(np.mean([ba('lcca', s, 'norm_only') - ba('lcca', s, 'off') for s in SEEDS])); acc['A_fullft_lraa'].append(np.mean(At)); acc['B_fullft_lraa'].append(np.mean([ba('fullft_lraa', s, 'norm_only') - ba('fullft_lraa', s, 'off') for s in SEEDS]))
    acc['delta_A'].append(np.mean(Af) - np.mean(At)); acc['D2'].append(np.mean([ba('fullft_lraa', s, 'on') - ba('fullft_none', s, 'on') for s in SEEDS]))
BOOT = {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in acc.items()}
mcn = []
for s in SEEDS:
    for a, b in (('lcca', 'mlp'), ('lcca', 'token_mixer'), ('lcca', 'lora')):
        ca, cb = PRED(s, a) == DEV_GOLD, PRED(s, b) == DEV_GOLD; x, y = int((ca & ~cb).sum()), int((~ca & cb).sum()); mcn.append({'seed': s, 'comparison': f'{a}_vs_{b}', 'first_only': x, 'second_only': y, 'p_exact': exact_mcnemar_p(x, y)})
    for m in ('mlp', 'lcca', 'token_mixer', 'fullft_lraa'):
        for x_, y_ in (('on', 'norm_only'), ('norm_only', 'off')):
            ca, cb = PRED(s, m, x_) == DEV_GOLD, PRED(s, m, y_) == DEV_GOLD; x, y = int((ca & ~cb).sum()), int((~ca & cb).sum()); mcn.append({'seed': s, 'comparison': f'{m}_{x_}_vs_{y_}', 'first_only': x, 'second_only': y, 'p_exact': exact_mcnemar_p(x, y)})
    ca, cb = PRED(s, 'fullft_lraa') == DEV_GOLD, PRED(s, 'fullft_none') == DEV_GOLD; x, y = int((ca & ~cb).sum()), int((~ca & cb).sum()); mcn.append({'seed': s, 'comparison': 'fullft_lraa_vs_none', 'first_only': x, 'second_only': y, 'p_exact': exact_mcnemar_p(x, y)})
pd.DataFrame(mcn).to_csv(RESULT_DIR / 'mcnemar.csv', index=False)
label = {'head': 'Head-only', 'mlp': 'Token-wise MLP', 'lcca': 'LRAA', 'token_mixer': 'Token mixer', 'lora': 'LoRA r = 11', 'fullft_lraa': 'Full FT + LRAA', 'fullft_none': 'Full FT, no adapter'}
lines = ['| Method | on | norm_only | off | on − off [seed CI] | A [seed CI] | B [seed CI] |', '|---|---:|---:|---:|---|---|---|']
for m in ALL_METHODS:
    mm = means[m]; on = f"{mm['on']:.2f}"; no = f"{mm['norm_only']:.2f}" if 'norm_only' in mm else '—'; off = f"{mm['off']:.2f}" if 'off' in mm else '—'
    oo = f"{Q['on_off_'+m]['mean']:+.2f} [{Q['on_off_'+m]['ci95_low']:+.2f}, {Q['on_off_'+m]['ci95_high']:+.2f}]" if f'on_off_{m}' in Q else '—'
    A_ = f"{Q['A_'+m]['mean']:+.2f} [{Q['A_'+m]['ci95_low']:+.2f}, {Q['A_'+m]['ci95_high']:+.2f}]" if f'A_{m}' in Q else '—'; B_ = f"{Q['B_'+m]['mean']:+.2f} [{Q['B_'+m]['ci95_low']:+.2f}, {Q['B_'+m]['ci95_high']:+.2f}]" if f'B_{m}' in Q else '—'
    lines.append(f'| {label[m]} | {on} | {no} | {off} | {oo} | {A_} | {B_} |')
lines += ['', '| Contrast | Mean [seed CI] | Cluster CI | Positive seeds |', '|---|---|---|:---:|']
for k, lab in (('lcca_minus_mlp', 'LRAA − MLP'), ('lcca_minus_mixer', 'LRAA − mixer'), ('lcca_minus_lora', 'LRAA − LoRA'), ('delta_A', 'ΔA = A_frozen − A_FT'), ('delta_on_off', 'Δ(on−off) frozen − FT'), ('D2', 'Full FT + LRAA − no adapter'), ('G_lcca', 'G = LRAA − head-only')):
    q = Q[k]; cb = f"[{BOOT[k][0]:+.2f}, {BOOT[k][1]:+.2f}]" if k in BOOT else '—'; lines.append(f"| {lab} | {q['mean']:+.2f} [{q['ci95_low']:+.2f}, {q['ci95_high']:+.2f}] | {cb} | {q['positive_seeds']}/{len(SEEDS)} |")
TABLE = '\n'.join(lines); (RESULT_DIR / 'winogrande_table.md').write_text(TABLE + '\n')
summary = {'manifest': MANIFEST_ID, 'smoke_test': SMOKE_TEST, 'seeds': SEEDS, 'means': means, 'seed_level': Q, 'cluster_bootstrap_95': BOOT, 'decisions': DEC, 'optimization_gate': gate, 'param_counts': PARAM_COUNTS, 'dev_twin_clusters': int(N_CLUSTERS)}
(RESULT_DIR / 'winogrande_analysis_summary.json').write_text(json.dumps(summary, indent=2, default=float))
print(TABLE); print('\nDecisions:', json.dumps(DEC, indent=1, default=str)); display(RUN.pivot_table(index=['method', 'seed'], columns='state', values='balanced_accuracy').mul(100).round(2))

In [ ]:
# Environment, manifest, archive, Drive copy, download.
env = {'notebook': 'NB-5 W-1 WinoGrande', 'manifest': MANIFEST_ID, 'smoke_test': SMOKE_TEST, 'gpu': GPU_NAME, 'amp_dtype': str(AMP_DTYPE), 'torch': torch.__version__, 'cuda': torch.version.cuda,
       'transformers': __import__('transformers').__version__, 'winogrande_archive_sha256': WG_SHA, 'seeds': SEEDS, 'epochs': EPOCHS, 'bootstrap_seed': BOOTSTRAP_SEED, 'run_started_utc': RUN_STARTED_UTC,
       'run_finished_utc': datetime.now(timezone.utc).isoformat(), 'session_active_seconds': time.perf_counter() - RUN_STARTED_PERF, 'state_files': {f'{m}_seed{s}': {k: R[(s, m)].get(k) for k in ('state_sha256', 'state_bytes')} for s in SEEDS for m in ALL_METHODS}}
(RESULT_DIR / 'run_environment.json').write_text(json.dumps(env, indent=2))
for s in SEEDS:
    for m in ALL_METHODS: shutil.copy2(prog_path(s, m), RESULT_DIR / f'run_{m}_seed{s}.json')
rows_ = [{'file': p.name, 'bytes': p.stat().st_size, 'sha256': sha256_file(p)} for p in sorted(RESULT_DIR.iterdir()) if p.is_file() and p.name != 'FILE_MANIFEST_SHA256.csv']
pd.DataFrame(rows_).to_csv(RESULT_DIR / 'FILE_MANIFEST_SHA256.csv', index=False)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); zip_path = Path(shutil.make_archive(f"/content/fmi_nb5_winogrande_{'smoke_' if SMOKE_TEST else ''}results_{stamp}", 'zip', RESULT_DIR))
with zipfile.ZipFile(zip_path) as zf:
    if zf.testzip(): raise RuntimeError('ZIP corrupt')
shutil.copy2(zip_path, DRIVE_ROOT / zip_path.name); print('Result ZIP:', zip_path, '| sha256', sha256_file(zip_path))
from google.colab import files
files.download(str(zip_path))